Import libraries


In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Tarnet/main_tarnet.ipynb")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 19.5 ms, sys: 7.99 ms, total: 27.5 ms
Wall time: 26.9 ms
CPU times: user 10.7 ms, sys: 2.02 ms, total: 12.7 ms
Wall time: 12.6 ms
CPU times: user 3.74 ms, sys: 1 ms, total: 4.74 ms
Wall time: 4.74 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:10].astype(float))

y_train[:10] [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [8]:
epochs = 70
lr = 5e-4
wd = 1e-5
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 15
shared_dropout = 0
outcome_droupout = 0
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")
print (f" activation = {activation}")

 epochs = 70
 learning rate = 0.0005
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 15
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30
 activation = <class 'torch.nn.modules.activation.ReLU'>


In [10]:
import pandas as pd
import numpy as np
import torch
from itertools import product

# 1. Grid search config
seeds = [412312, 42, 1874, 902745, 1]
lr_grid = [3e-5, 5e-5, 1e-4, 3e-4, 5e-4, 1e-3]
wd_grid = [0.0, 1e-6, 1e-5, 1e-4]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
grid_results = []

print(f"🔄 Bắt đầu grid search: {len(lr_grid)} x {len(wd_grid)} = {len(lr_grid) * len(wd_grid)} cấu hình")

# 2. Loop over all (lr, wd) pairs
for grid_lr, grid_wd in product(lr_grid, wd_grid):
    run_rows = []
    
    print("\n" + "=" * 95)
    print(f"Cấu hình: lr={grid_lr:.1e}, weight_decay={grid_wd:.1e}")
    print("=" * 95)

    for SEED in seeds:
        seed_everything(SEED)

        # Reinitialize model for each seed
        tarnet = Tarnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_droupout=outcome_droupout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        tarnet.fit(train_loader, val_loader)

        # Validation prediction
        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p = tarnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

        # Test prediction
        x_test_device = x_men_test_t.to(device)
        y0_pred, y1_pred = tarnet.predict(x_test_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_test_t.detach().cpu().numpy().flatten()
        t_true = t_men_test_t.detach().cpu().numpy().flatten()

        # ATE error
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

        row = {
            "Seed": SEED,
            "lr": grid_lr,
            "weight_decay": grid_wd,
            "Val_AUQC": current_val_auqc,
            "AUUC": auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "AUQC": auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "Lift": lift(y_true, t_true, uplift_pred, h=0.3),
            "KRCC": krcc(y_true, t_true, uplift_pred, bins=100),
            "ATE_Err": abs(ate_pred - ate_true)
        }
        run_rows.append(row)
        
        print(f"  ✔ Seed {SEED} | Val_AUQC={current_val_auqc:.4f} | Test_AUQC={row['AUQC']:.4f}")

    # Aggregate each hyperparameter pair
    df_pair = pd.DataFrame(run_rows)
    mean_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].mean()
    std_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].std()

    grid_results.append({
        "lr": grid_lr,
        "weight_decay": grid_wd,
        "mean_Val_AUQC": mean_pair["Val_AUQC"],
        "mean_AUUC": mean_pair["AUUC"],
        "mean_AUQC": mean_pair["AUQC"],
        "mean_Lift": mean_pair["Lift"],
        "mean_KRCC": mean_pair["KRCC"],
        "mean_ATE_Err": mean_pair["ATE_Err"],
        "std_Val_AUQC": std_pair["Val_AUQC"],
        "std_AUUC": std_pair["AUUC"],
        "std_AUQC": std_pair["AUQC"],
        "std_Lift": std_pair["Lift"],
        "std_KRCC": std_pair["KRCC"],
        "std_ATE_Err": std_pair["ATE_Err"]
    })

# 3. Final ranking by mean test AUQC
df_grid = pd.DataFrame(grid_results).sort_values("mean_AUQC", ascending=False).reset_index(drop=True)
best_cfg = df_grid.iloc[0]

print("\n" + "=" * 110)
print(f"{'KẾT QUẢ GRID SEARCH (XẾP HẠNG THEO MEAN TEST AUQC)':^110}")
print("=" * 110)

display_cols = [
    "lr", "weight_decay",
    "mean_Val_AUQC", "mean_AUUC", "mean_AUQC", "mean_Lift", "mean_KRCC", "mean_ATE_Err",
    "std_Val_AUQC", "std_AUUC", "std_AUQC", "std_Lift", "std_KRCC", "std_ATE_Err"
]

print(df_grid[display_cols].to_string(
    index=False,
    formatters={
        "lr": "{:.1e}".format,
        "weight_decay": "{:.1e}".format,
        "mean_Val_AUQC": "{:.4f}".format,
        "mean_AUUC": "{:.4f}".format,
        "mean_AUQC": "{:.4f}".format,
        "mean_Lift": "{:.4f}".format,
        "mean_KRCC": "{:.4f}".format,
        "mean_ATE_Err": "{:.4f}".format,
        "std_Val_AUQC": "{:.4f}".format,
        "std_AUUC": "{:.4f}".format,
        "std_AUQC": "{:.4f}".format,
        "std_Lift": "{:.4f}".format,
        "std_KRCC": "{:.4f}".format,
        "std_ATE_Err": "{:.4f}".format
    }
))

print("-" * 110)
print("Best hyperparameters by mean TEST AUQC:")
print(f"  lr={best_cfg['lr']:.1e}, weight_decay={best_cfg['weight_decay']:.1e}")
print(f"  mean TEST AUQC = {best_cfg['mean_AUQC']:.4f} ± {best_cfg['std_AUQC']:.4f}")
print("=" * 110)

🔄 Bắt đầu grid search: 6 x 4 = 24 cấu hình

Cấu hình: lr=3.0e-05, weight_decay=0.0e+00
Locked random seed: 412312
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/70 | Loss: 1.4047 | Val Loss: 1.4035 | Val Qini: 0.5968 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5968 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Loss: 1.4002 | Val Loss: 1.3993 | Val Qini: 0.5831 (patience: 1/15)EMA Trend: 0.5947 | (patience: 1/15)
Epoch 3/70 | Loss: 1.3962 | Val Loss: 1.3952 | Val Qini: 0.5690 (patience: 2/15)EMA Trend: 0.5909 | (patience: 2/15)
Epoch 4/70 | Loss: 1.3923 | Val Loss: 1.3912 | Val Qini: 0.5497 (patience: 3/15)EMA Trend: 0.5847 | (patience: 3/15)
Epoch 5/70 | Loss: 1.3881 | Val Loss: 1.3872 | Val Qini: 0.5373 (patience: 4/15)EMA Trend: 0.5776 | (patience: 4/15)
Epoch 6/70 | Loss: 1.3846 

In [ ]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    tarnet = Tarnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr, 
        weight_decay=5e-6, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_droupout=outcome_droupout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    tarnet.fit(train_loader, val_loader)
    
    # --- Predict on VALIDATION set ---
    # Giả sử bạn có x_men_val_t, y_men_val_t, t_men_val_t từ val_loader hoặc tương đương
    x_val_device = x_men_val_t.to(device)
    y0_val_p, y1_val_p = tarnet.predict(x_val_device)
    uplift_val = (y1_val_p - y0_val_p).cpu().numpy().flatten()
    y_val_true_np = y_men_val_t.cpu().numpy().flatten()
    t_val_true_np = t_men_val_t.cpu().numpy().flatten()
    
    current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

    # --- Predict on TEST set ---
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = tarnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán ATE
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'Val_AUQC': current_val_auqc, # Thêm Val AUQC vào đây
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED} | Val AUQC: {current_val_auqc:.4f}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

# Đưa cột Val_AUQC lên đầu cho dễ nhìn
cols = ['Seed', 'Val_AUQC', 'AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']
df_results = df_results[cols]

print("\n" + "="*95)
print(f"{'CHI TIẾT TỪNG SEED (VALIDATION & TEST)':^95}")
print("="*95)
print(df_results.to_string(index=False, formatters={
    'Val_AUQC': '{:,.4f}'.format, 'AUUC': '{:,.4f}'.format, 
    'AUQC': '{:,.4f}'.format, 'Lift': '{:,.4f}'.format, 
    'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*95)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^95}")
print("-" * 95)
for metric in ['Val_AUQC', 'AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*95)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/70 | Loss: 1.3621 | Val Loss: 1.3494 | Val Qini: 0.3630 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3630 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Loss: 1.3067 | Val Loss: 1.2895 | Val Qini: 0.2603 (patience: 1/20)EMA Trend: 0.3476 | (patience: 1/20)
Epoch 3/70 | Loss: 1.2185 | Val Loss: 1.1867 | Val Qini: 0.3400 (patience: 2/20)EMA Trend: 0.3464 | (patience: 2/20)
Epoch 4/70 | Loss: 1.0434 | Val Loss: 0.9786 | Val Qini: 0.3175 (patience: 3/20)EMA Trend: 0.3421 | (patience: 3/20)
Epoch 5/70 | Loss: 0.7207 | Val Loss: 0.6260 | Val Qini: 0.5833 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3783 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/70 | Loss: 0.3457 | Val